# 1. Chạy luồng chuẩn hóa Kỹ năng (JD & CV) & Khớp mờ Doanh nghiệp
Notebook này thực thi thuật toán chuẩn hóa thực tế của hệ thống đối với kỹ năng thô (từ JD và CV) và tên công ty thô.


## Bước 1.1: Nhập thư viện và thiết lập đường dẫn


In [7]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Ép PyTorch chạy CPU để tránh lỗi DLL WinError 1114 trong môi trường Jupyter của VS Code
import json
import sys
import importlib.util
from pathlib import Path
from dotenv import load_dotenv

workspace_root = Path(r'f:\HCMUS_KH\LuanVan\JobVisualization_BE')
script_dir = workspace_root / 'KiemThu' / 'KiemThu_SkillNormalization'
normalize_script_path = workspace_root / 'Db' / 'pipeline' / 'normalize' / '2_1_normalized_data' / 'normalize_pipeline_v2.py'
import_script_path = workspace_root / 'Db' / 'pipeline' / 'import' / '3_import' / 'import.py'
matching_cv_path = workspace_root / 'matching_cv'

# Nạp động module normalize_pipeline_v2.py
spec_norm = importlib.util.spec_from_file_location('normalize_pipeline_v2', str(normalize_script_path))
norm_mod = importlib.util.module_from_spec(spec_norm)
sys.path.insert(0, str(normalize_script_path.parent))
sys.path.insert(0, str(workspace_root))
spec_norm.loader.exec_module(norm_mod)

# Nạp động module import.py
spec_imp = importlib.util.spec_from_file_location('import_module', str(import_script_path))
imp_mod = importlib.util.module_from_spec(spec_imp)
sys.path.insert(0, str(import_script_path.parent))
spec_imp.loader.exec_module(imp_mod)

# Nạp động module normalizer.py của matching_cv
spec_cv = importlib.util.spec_from_file_location('matching_cv_normalizer', str(matching_cv_path / 'normalizer.py'))
cv_norm_mod = importlib.util.module_from_spec(spec_cv)
sys.path.insert(0, str(matching_cv_path))
spec_cv.loader.exec_module(cv_norm_mod)

print('✓ Đã import thành công các module từ pipeline hệ thống (bao gồm cả matching_cv)!')


✓ Đã import thành công các module từ pipeline hệ thống (bao gồm cả matching_cv)!


## Bước 1.2: Kết nối cơ sở dữ liệu và tải từ điển


In [8]:
dotenv_path = workspace_root / 'Db' / '.env'
load_dotenv(dotenv_path, override=True)
db_url = os.environ.get('DATABASE_URL')
if not db_url:
    raise ValueError('DATABASE_URL không được cấu hình')

print('🔄 Đang tải từ điển kỹ năng từ database...')
skills = norm_mod.load_dictionary_from_db(db_url, 'skills')
benefits = norm_mod.load_dictionary_from_db(db_url, 'benefits')
db_skills_by_name = {row[1]: row[0] for row in skills}

skills_with_meta = norm_mod.load_skills_metadata_from_db(db_url, 'skills')
allowed_types = {'Hard Skill', 'Specialized Skill', 'Common skill', 'Common Skill'}
lightcast_skills = []
lightcast_metadata = {}

for sid, name, cat, stype in skills_with_meta:
    lightcast_metadata[name] = {
        'subcategory': cat,
        'type': stype
    }
    if stype in allowed_types and name:
        lightcast_skills.append(name)

seen = set()
lightcast_skills_dedup = []
for s in lightcast_skills:
    if s not in seen:
        seen.add(s)
        lightcast_skills_dedup.append(s)
lightcast_skills = lightcast_skills_dedup

norm_mod.GLOBAL_LIGHTCAST_METADATA = lightcast_metadata

lightcast_skill_map = []
for name in lightcast_skills:
    sid = db_skills_by_name.get(name)
    if sid is not None:
        lightcast_skill_map.append((sid, name))
    else:
        lightcast_skill_map.append((-1, name))

print(f'✓ Từ điển tải thành công: {len(lightcast_skills)} kỹ năng tiêu chuẩn.')


🔄 Đang tải từ điển kỹ năng từ database...
✓ Từ điển tải thành công: 6092 kỹ năng tiêu chuẩn.


## Bước 1.3: Khởi tạo mô hình SentenceTransformer và Vector Cache


In [9]:
from sentence_transformers import SentenceTransformer
import pickle

print('🔄 Đang khởi tạo SentenceTransformer (all-MiniLM-L6-v2)...')
model = SentenceTransformer('all-MiniLM-L6-v2')

cache_dir = normalize_script_path.parent / 'cache'
lightcast_cache_file = cache_dir / 'lightcast_embeddings_minilm.pkl'
benefits_cache_file = cache_dir / 'benefits_embedding.pkl'

if lightcast_cache_file.exists():
    print('🔄 Đang nạp FAISS embedding cache cho kỹ năng...')
    with open(lightcast_cache_file, 'rb') as fh:
        lc_cache = pickle.load(fh)
    lightcast_emb = lc_cache.get('emb')
else:
    print('🔄 Đang tạo mới embedding vectors cho kỹ năng...')
    lightcast_emb = norm_mod.compute_embeddings(model, lightcast_skills)

benefit_names = [b[1] for b in benefits]
if benefits_cache_file.exists():
    with open(benefits_cache_file, 'rb') as fh:
        be_cache = pickle.load(fh)
    benefits_emb = be_cache.get('emb')
else:
    benefits_emb = norm_mod.compute_embeddings(model, benefit_names)

skills_cache_path = cache_dir / 'mapped_skills_cache.json'
mapping_cache = {}
if skills_cache_path.exists():
    with skills_cache_path.open('r', encoding='utf-8') as fh:
        mapping_cache = json.load(fh)

# Load pre-labeled types
labeled_skills_path = normalize_script_path.parent / 'raw_extracted_skills_fixed_type.csv'
if labeled_skills_path.exists():
    import csv
    with open(labeled_skills_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            skill = row.get('Raw Skill', '').strip()
            skill_type = row.get('Type', '').strip().lower()
            if skill:
                norm_mod.GLOBAL_LABELED_SKILL_TYPES[skill] = skill_type

print('✓ Đã nạp thành công mô hình và toàn bộ vector embeddings.')


🔄 Đang khởi tạo SentenceTransformer (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13351.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Đang nạp FAISS embedding cache cho kỹ năng...
✓ Đã nạp thành công mô hình và toàn bộ vector embeddings.


## Bước 1.4: Chạy luồng chuẩn hóa cho JDs tuyển dụng & Doanh nghiệp


In [10]:
import psycopg2

input_jds_file = script_dir / 'extracted_jds.json'
with open(input_jds_file, 'r', encoding='utf-8') as f:
    jobs = json.load(f)

print(f'🔄 Bắt đầu chuẩn hóa cho {len(jobs)} JDs...')

conn = psycopg2.connect(db_url)
cur = conn.cursor()

actual_skills = []
actual_companies = []

for item in jobs:
    url = item.get('job_url') or item.get('url')
    skills_in = item.get('extracted_skills') or []
    raw_company = item.get('company_name') or item.get('company', {}).get('name') or ''
    
    # 1. Chuẩn hóa Kỹ năng JD
    job_item = {
        'url': url,
        'title': item.get('title', 'Software Engineer'),
        'extracted_skills': [{'skill_name': s} if isinstance(s, str) else s for s in skills_in]
    }
    
    normalized_job = norm_mod.normalize_job(
        job=job_item,
        skill_names=lightcast_skills,
        skill_emb=lightcast_emb,
        skill_map=lightcast_skill_map,
        benefit_names=benefit_names,
        benefit_emb=benefits_emb,
        benefit_map=benefits,
        model=model,
        threshold=0.5,
        top_k=10,
        disable_llm_rerank=True,
        mapping_cache=mapping_cache
    )
    normalized_job = norm_mod.remove_unmapped_items(normalized_job)
    
    mapped_skills = normalized_job.get('normalized_skills', [])
    unmapped_skills = normalized_job.get('unmatched_skills', [])
    
    norm_map = {}
    for s_entry in mapped_skills:
        norm_map[s_entry['original']] = s_entry['mapped_name']
    for s_entry in unmapped_skills:
        norm_map[s_entry['original']] = None
        
    for s in skills_in:
        s_name = s if isinstance(s, str) else s.get('skill_name')
        actual_skills.append({
            'url': url,
            'skill_extract': s_name,
            'skill_normalize': norm_map.get(s_name, None)
        })
        
    # 2. Khớp mờ Doanh nghiệp
    similar_id = imp_mod.find_similar_company(cur, raw_company)
    if similar_id:
        cur.execute('SELECT name FROM companies WHERE company_id = %s', (similar_id,))
        similar_name = cur.fetchone()[0]
    else:
        similar_name = None
        
    actual_companies.append({
        'url': url,
        'company_raw': raw_company,
        'company_normalize': similar_name
    })

cur.close()
conn.close()
print('✓ Đã chuẩn hóa xong phần JDs và Doanh nghiệp.')


🔄 Bắt đầu chuẩn hóa cho 24 JDs...
✓ Đã chuẩn hóa xong phần JDs và Doanh nghiệp.


## Bước 1.5: Chạy luồng chuẩn hóa kỹ năng cho CVs sinh viên


In [11]:
input_cvs_file = workspace_root / 'KiemThu' / 'KiemThu_LLM_Extract' / 'pipeline_cvs_output.json'
with open(input_cvs_file, 'r', encoding='utf-8') as f:
    cvs = json.load(f)

print(f'🔄 Bắt đầu chuẩn hóa kỹ năng cho {len(cvs)} CVs...')
actual_cv_skills = []

for item in cvs:
    filename = item.get('filename')
    skills_in = item.get('skills', [])
    
    # Gọi trực tiếp module chuẩn hóa cv của matching_cv
    normalized_skills = cv_norm_mod.normalize_student_skills(
        raw_skills=skills_in,
        model_name='all-MiniLM-L6-v2',
        top_k=10,
        disable_llm_rerank=True
    )
    
    norm_map = {s_entry['original']: s_entry['mapped_name'] for s_entry in normalized_skills}
    
    for s in skills_in:
        actual_cv_skills.append({
            'filename': filename,
            'skill_extract': s,
            'skill_normalize': norm_map.get(s, None)
        })

# Lưu kết quả chạy thực tế ra file JSON
output_file = script_dir / 'actual_normalization_results.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump({
        'skills': actual_skills,
        'cv_skills': actual_cv_skills,
        'companies': actual_companies
    }, f, ensure_ascii=False, indent=2)

print(f'✅ Đã lưu kết quả chuẩn hóa thực tế của cả JD và CV vào: {output_file.name}')


🔄 Bắt đầu chuẩn hóa kỹ năng cho 20 CVs...


Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing contextual embeddings for 6092 Main FAISS items...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7545.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22867.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8478.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embed

✅ Đã lưu kết quả chuẩn hóa thực tế của cả JD và CV vào: actual_normalization_results.json
